# Transcript Matching — Method & Preprocessing Evaluation
Accuracy (5-fold cross-validation) of each fuzzy-ratio feature set under every transcript preprocessing (16 combinations of the 4 preprocessing flags x 6 cut lengths), on the labeled pairs used to train the matching model.

Computed by `python -m scripts.evaluate_matching_methods` (run it first); this notebook only shows the results and the LaTeX table.

In [1]:
import os

import pandas as pd

if os.path.isdir("../data/"):
    os.chdir("../")

from scripts.evaluate_matching_methods import (CUTS, FEATURE_SETS, OUTPUT_DIR, PREPROCESSING_FLAGS,
                                               PRODUCTION_FEATURE_SET, PRODUCTION_VARIANT, VARIANTS)

results = pd.read_csv(f"{OUTPUT_DIR}/matching_evaluation.csv")
results["preprocessing"] = results["variant"].str.rsplit("_cut", n=1).str[0]
PREPROCESSINGS = list(dict.fromkeys(v.rsplit("_cut", 1)[0] for v in VARIANTS))
print(f"{results.n_pairs.iloc[0]} labeled pairs, {results.variant.nunique()} variants, "
      f"pipeline: {PRODUCTION_VARIANT} with {PRODUCTION_FEATURE_SET}")

303 labeled pairs, 96 variants, pipeline: strip_accents-lowercase-remove_brackets_cut2000 with partial sort set


## 1. Best configurations

In [2]:
cols = ["preprocessing", "cut_transcripts", "feature_set", "mean_accuracy", "std_accuracy"]
best = results.sort_values("mean_accuracy", ascending=False)[cols + ["is_production"]].head(15)
best.style.format({"mean_accuracy": "{:.3f}", "std_accuracy": "{:.3f}"})

,preprocessing,cut_transcripts,feature_set,mean_accuracy,std_accuracy,is_production
81,strip_accents,20000,partial set,0.993,0.013,False
635,strip_accents-lowercase-remove_brackets-remove_openers,0,sort set,0.993,0.008,False
509,strip_accents-lowercase-remove_openers,0,sort set,0.993,0.008,False
333,strip_accents-remove_openers,20000,partial set,0.993,0.013,False
215,strip_accents-lowercase,0,sort set,0.990,0.008,False
88,lowercase,0,partial set,0.990,0.013,False
39,none,20000,partial set,0.990,0.013,False
181,remove_openers,2000,partial sort set,0.990,0.013,False
46,strip_accents,0,partial set,0.990,0.013,False
47,strip_accents,0,sort set,0.990,0.008,False


## 2. Preprocessing x cut, for one feature set

In [3]:
FEATURE_SET = PRODUCTION_FEATURE_SET   # any key of FEATURE_SETS

sub = results[results.feature_set == FEATURE_SET]
grid = sub.pivot(index="preprocessing", columns="cut_transcripts", values="mean_accuracy").loc[PREPROCESSINGS, CUTS]
grid.style.background_gradient(cmap="Greens", axis=None).format("{:.3f}")

cut_transcripts,0,2000,5000,10000,15000,20000
preprocessing,,,,,,
none,0.987,0.990,0.983,0.987,0.990,0.983
strip_accents,0.990,0.977,0.970,0.970,0.984,0.980
lowercase,0.990,0.980,0.980,0.974,0.974,0.977
remove_brackets,0.987,0.984,0.987,0.987,0.990,0.987
remove_openers,0.987,0.990,0.983,0.987,0.990,0.983
strip_accents-lowercase,0.990,0.970,0.964,0.947,0.964,0.970
strip_accents-remove_brackets,0.990,0.974,0.970,0.967,0.984,0.977
strip_accents-remove_openers,0.990,0.980,0.970,0.970,0.980,0.980
lowercase-remove_brackets,0.990,0.980,0.980,0.974,0.974,0.977


## 3. Feature set x cut, for one preprocessing

In [4]:
PREPROCESSING = PRODUCTION_VARIANT.rsplit("_cut", 1)[0]   # any of PREPROCESSINGS

sub = results[results.preprocessing == PREPROCESSING]
def table(values):
    return sub.pivot(index="feature_set", columns="cut_transcripts", values=values).loc[list(FEATURE_SETS), CUTS]

evaluate_df_results = table("mean_accuracy")
std_df_results = table("std_accuracy")
evaluate_df_results.style.highlight_max(color='lightgreen', axis=0).format("{:.3f}")

cut_transcripts,0,2000,5000,10000,15000,20000
feature_set,,,,,,
partial,0.977,0.980,0.980,0.957,0.980,0.977
sort,0.406,0.637,0.554,0.522,0.482,0.547
set,0.984,0.947,0.947,0.937,0.957,0.974
partial sort,0.977,0.980,0.980,0.964,0.980,0.984
partial set,0.984,0.970,0.974,0.957,0.974,0.987
sort set,0.990,0.954,0.941,0.937,0.950,0.987
partial sort set,0.990,0.967,0.964,0.944,0.964,0.967


In [5]:
std_df_results.style.format("{:.3f}")

cut_transcripts,0,2000,5000,10000,15000,20000
feature_set,,,,,,
partial,0.017,0.012,0.012,0.035,0.019,0.017
sort,0.029,0.050,0.015,0.035,0.058,0.082
set,0.010,0.013,0.019,0.016,0.008,0.008
partial sort,0.017,0.012,0.012,0.033,0.019,0.021
partial set,0.010,0.012,0.013,0.028,0.013,0.012
sort set,0.008,0.016,0.013,0.012,0.018,0.019
partial sort set,0.013,0.010,0.012,0.027,0.019,0.023


### LaTeX table (feature set x cut, for the preprocessing above)

In [6]:
label_map = {
    'partial': 'pr', 'sort': 'srtr', 'set': 'str',
    'partial sort': 'pr \\& srtr', 'partial set': 'pr \\& str',
    'sort set': 'srtr \\& str', 'partial sort set': 'pr, srtr \\& str',
}
std_stars = std_df_results.map(lambda x: "$^*$" if x < 0.05 else "")
evaluate_complete = (
    evaluate_df_results.map(lambda x: f"{x:.2f}") + std_stars
).rename(index=label_map)

for i, (idx, row) in enumerate(evaluate_complete.iterrows()):
    if i in [3, 6]:
        print('\\hline')
    cells = [
        f"\\textbf{{{v}}}" if '0.99' in v else v
        for v in row.values
    ]
    print(idx + ' & ' + ' & '.join(cells) + '\\\\')

pr & 0.98$^*$ & 0.98$^*$ & 0.98$^*$ & 0.96$^*$ & 0.98$^*$ & 0.98$^*$\\
srtr & 0.41$^*$ & 0.64 & 0.55$^*$ & 0.52$^*$ & 0.48 & 0.55\\
str & 0.98$^*$ & 0.95$^*$ & 0.95$^*$ & 0.94$^*$ & 0.96$^*$ & 0.97$^*$\\
\hline
pr \& srtr & 0.98$^*$ & 0.98$^*$ & 0.98$^*$ & 0.96$^*$ & 0.98$^*$ & 0.98$^*$\\
pr \& str & 0.98$^*$ & 0.97$^*$ & 0.97$^*$ & 0.96$^*$ & 0.97$^*$ & \textbf{0.99$^*$}\\
srtr \& str & \textbf{0.99$^*$} & 0.95$^*$ & 0.94$^*$ & 0.94$^*$ & 0.95$^*$ & \textbf{0.99$^*$}\\
\hline
pr, srtr \& str & \textbf{0.99$^*$} & 0.97$^*$ & 0.96$^*$ & 0.94$^*$ & 0.96$^*$ & 0.97$^*$\\
